# Introduction


In this notebook we introduce a lightweight approach to using open source language models for zero-shot classification - assigning text to categories without task-specific training.

We'll use **Specialized Zero-Shot Classifiers**: Modern transformer models like BART that can classify text without any task-specific training. This notebook uses CPU-compatible models suitable for running on any machine.

**Classification** is the act of assigning a probability between 0 and 1 to label(s) being applicable to text.

For example: "The sun is out today!" seems happy - a human might score it 90% on "happiness". But "It is raining outside.." might score only 10%. Modern language models can make these assessments automatically based on their deep understanding of language.

Let's get started!


## Installation

If you haven't already, install the required packages:

```bash
pip install -r requirements.txt
```

Or install individually:

```bash
pip install transformers>=4.30.0 torch>=2.0.0 sentencepiece>=0.1.99
```


# Setup


In [ ]:
from transformers import pipeline
import torch

In [ ]:
c = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1,
)

## About the Model

We're using BART-large-MNLI, a reliable zero-shot classifier:

- **CPU-Compatible**: Works on CPU without requiring GPU (though it may be slower)
- **Proven Performance**: Well-tested model for zero-shot classification tasks
- **Widely Available**: Publicly accessible on HuggingFace
- **Size**: Around 400M parameters - not the smallest, but very capable


# Approach 1: Specialized Zero-Shot Classifier

## Testing whether a label is applicable


Let's see if we can replicate the ideas above. Is the sentence "The sun is out today!" indeed classifiably happy?

To do so, we use the defined interface of our classifier, which takes (at least) the following arguments: a sentence, a list of labels.

Perhaps it's easier to just look at an example!


In [ ]:
c("The sun is out today.", ["Happy"])

{'sequence': 'The sun is out today.',
 'labels': ['Happy'],
 'scores': [0.958601176738739]}

Great result! It looks like our sentence is indeed very happy. With a score of almost 96%, it doesn't get much happier than that.

Challenge: Can you find a way to modify the sentence such that it is even happier?


In [ ]:
# Modify the sentence to something even happier
c("The sun is out today.", ["Happy"])

{'sequence': 'The sun is out today.',
 'labels': ['Happy'],
 'scores': [0.958601176738739]}

And how about our "unhappy" sentence, will it indeed agree?


In [ ]:
c("It is raining outside..", ["Happy"])

{'sequence': 'It is raining outside..',
 'labels': ['Happy'],
 'scores': [0.00019142446399200708]}

Nobody likes rain..


# Multiple labels


If that's not a happy sentence, then perhaps we can conclude it's a sad one? Does our model agree?

We can ask it to make a choice between several available labels.


In [ ]:
c("It is raining outside..", ["Happy", "Sad"])

{'sequence': 'It is raining outside..',
 'labels': ['Sad', 'Happy'],
 'scores': [0.9559348821640015, 0.04406508430838585]}

The mathematics of the model work out slightly differently to before for multi-labeling, but the conclusion remains the same, it certainly considers the sentence to be much more sad than happy.

What about other labels, like wet, dry, high, low and colorful or gray?


c('It is raining outside..', ['Happy', 'Sad', 'Wet', 'Dry', 'High', 'Low', 'Colorful', 'Gray'])


Hard question, but a clear and agreeable answer, it's more Wet than any other of those labels. But isn't rain both wet and sad?

To answer that question we can ask the model to assign a probability per label, rather than forcing it to make a choice. Note that in the above result, all label probabilities add up to 100%.


In [ ]:
c(
    "It is raining outside..",
    ["Happy", "Sad", "Wet", "Dry", "High", "Low", "Colorful", "Gray"],
    multi_label=True,
)

{'sequence': 'It is raining outside..',
 'labels': ['Wet', 'Sad', 'Low', 'Gray', 'High', 'Colorful', 'Dry', 'Happy'],
 'scores': [0.9990817308425903,
  0.8136840462684631,
  0.7355218529701233,
  0.41334474086761475,
  0.03862270340323448,
  0.0017680968157947063,
  0.0003273721958976239,
  0.00019142446399200708]}

This makes sense!


# Edge cases


Negatiation. Surely, if rain is bad, then the opposite should be good?


In [ ]:
c("It is not raining outside..", ["Happy", "Sad"])

{'sequence': 'It is not raining outside..',
 'labels': ['Happy', 'Sad'],
 'scores': [0.8534077405929565, 0.14659228920936584]}

It depends on the context though, and the model can handle that pretty well too.


In [ ]:
c("After the drought it is finally raining outside..", ["Happy", "Sad"])

{'sequence': 'After the drought it is finally raining outside..',
 'labels': ['Happy', 'Sad'],
 'scores': [0.7387931942939758, 0.2612067759037018]}

# Multiple sentences/strings


We can also ask the model to classify multiple senteces in one go. That might be useful when you use this model for your trading strategy later.


In [ ]:
c(["The sun is out today.", "It is raining outside.."], ["Happy", "Sad"])

[{'sequence': 'The sun is out today.',
  'labels': ['Happy', 'Sad'],
  'scores': [0.9836673140525818, 0.016332656145095825]},
 {'sequence': 'It is raining outside..',
  'labels': ['Sad', 'Happy'],
  'scores': [0.9559348821640015, 0.04406508430838585]}]

# Approach 2: Using a Small Instruction-Tuned LLM (Optional)

This approach uses a small language model that can understand natural language instructions. This is more flexible but slower on CPU.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings

warnings.filterwarnings("ignore")

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded!")

In [ ]:
def classify_with_llm(text, labels):
    prompt = f"""Classify the following text into one of these categories: {', '.join(labels)}

Text: "{text}"

Answer with just the category name:"""

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = llm.generate(
        **inputs,
        max_new_tokens=10,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()

    return {"sequence": text, "predicted_label": response}


classify_with_llm("The sun is out today!", ["Happy", "Sad"])

### Try it with different examples


In [ ]:
classify_with_llm("It is raining outside..", ["Happy", "Sad", "Wet", "Dry"])

# Alternative: Lighter Model

If you need faster and more lightweight inference, you can try DistilBERT which is smaller and faster:


In [ ]:
c_small = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",
    device=-1,
)

c_small("The sun is out today!", ["Happy", "Sad"])

# Free-Form Exploration

Now it's your turn! Try different sentences, labels, and models to get a feel for what these classifiers can handle.
To explore additional open source models, visit [HuggingFace](https://huggingface.co/models).
